## *RaschPy* simulation functionality

This notebook works through examples of how to generate simulated data sets with `RaschPy` for experimental use where knowledge of the underlying 'ground truth' of the generating parameters is useful, for example when comparing the efficacy of different estimation algorithms, such as in Elliott & Buttery (2022a) or exploring the effect of fitting different Rasch models to the same data set, such as in Elliott & Buttery (2022b). There are separate classes for each model: `SLM_Sim` for the simple logistic model (or dichotomous Rasch model) (Rasch, 1960), `PCM_Sim` for the partial credit model (Masters, 1982), `RSM_Sim` for the rating scale model (Andrich, 1978), `MFRM_Sim_Global` for the many-facet Rasch model (Linacre, 1994), and the family of extended MFRMs (Elliott 2025, Elliott & Buttery, 2022b): `MFRM_Sim_Items` for the vector-by-item extended MFRM , `MFRM_Sim_Thresholds` for the vector-by-threshold extended MFRM, `MFRM_Sim_Matrix` for the matrix extended MFRM, and `MFRM_Sim_Bivector` for the bivector extended MFRM. All data is generated to fit the chosen model.

**References**

&nbsp;&nbsp;&nbsp;&nbsp; Andrich, D. (1978). A rating formulation for ordered response categories. *Psychometrika*, *43*(4), 561–573.

&nbsp;&nbsp;&nbsp;&nbsp;   Elliott, M. (2025). Extended many-facet Rasch models: Accounting for rater effects in automated essay scoring systems [Apollo - University of Cambridge Repository]. https://doi.org/10.17863/CAM.127567

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M., & Buttery, P. J. (2022a) Non-iterative Conditional Pairwise Estimation for the Rating Scale Model, *Educational and Psychological Measurement*, *82*(5), 989-1019.

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M. and Buttery, P. J. (2022b) Extended Rater Representations in the Many-Facet Rasch Model, *Journal of Applied Measurement*, *22*(1), 133-160.

&nbsp;&nbsp;&nbsp;&nbsp; Linacre, J. M. (1994). *Many-Facet Rasch Measurement*. MESA Press.

&nbsp;&nbsp;&nbsp;&nbsp; Masters, G. N. (1982). A Rasch model for partial credit scoring. *Psychometrika*, *47*(2), 149–174.

&nbsp;&nbsp;&nbsp;&nbsp; Rasch, G. (1960). *Probabilistic models for some intelligence and attainment tests*. Danmarks Pædagogiske
Institut.

Import the packages and set the working directory (here called `my_working_directory`) - you will save your output files here.

In [ ]:
import raschpy as rp
import numpy as np
import pandas as pd
import os

os.chdir('my_working_directory')

### `MFRM_Sim_Bivector`

Create an object `mfrm_sim_1` of the class `MFRM_Sim_Bivector` with randomised item locations, shared threshold set and person locations. In the bivector model, each rater's severity is decomposed into two additive components instead of one: a per-(rater, item) **item effect** (an overall leniency/severity profile across items) and a per-(rater, threshold) **threshold effect** (a category-width/consistency profile across thresholds, independent of overall leniency). `MFRM_Sim_Bivector` will generate both automatically when you pass `item_range`, `item_facet_range`, `threshold_facet_range`, `category_base`, `max_disorder`, `person_sd` and `offset` arguments to the simulation: item locations are sampled from a uniform distribution; person locations are sampled from a normal distribution. We pass `item_range=4` to have items covering a range of 4 logits, `item_facet_range=3` to have rater item-effects covering a range of 3 logits, and `threshold_facet_range=1` to have rater threshold-effects covering a range of 1 logit. We also pass `person_sd=2` and `offset=0.5` to have a sample of persons with a mean location 0.5 logits higher than the items, with a standard deviation of 2 logits, plus `category_base=1.5` and `max_disorder=1` (base category width of 1.5 logits, with random uniform variation controlled by `max_disorder`; a negative value for `max_disorder` permits disordered thresholds, hence the name). One other required argument is `max_score`, the maximum possible score for each item. There are 500 persons, 8 items and 10 raters, with no missing data for this simulation.

In [ ]:
mfrm_sim_1 = rp.MFRM_Sim_Bivector(no_of_items=8,
                                   no_of_persons=500,
                                   no_of_facet_elements=10,
                                   max_score=5,
                                   item_range=4,
                                   item_facet_range=3,
                                   threshold_facet_range=1,
                                   category_base=1.5,
                                   max_disorder=1,
                                   person_sd=2,
                                   offset=0.5,
                                   seed=42)

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_1.responses`, to file, and view the first 5 lines.

In [ ]:
mfrm_sim_1.responses.to_csv('mfrm_sim_1_responses.csv')
mfrm_sim_1.responses.head()

Save the generating item locations and Rasch-Andrich thresholds to file, and view them.

In [ ]:
mfrm_sim_1.items.to_csv('mfrm_sim_1_items.csv', header=None)
mfrm_sim_1.items.head()

In [ ]:
mfrm_sim_1.thresholds.to_csv('mfrm_sim_1_thresholds.csv', header=None)
mfrm_sim_1.thresholds

Unlike the other extended MFRM simulations, the bivector model stores its two rater-effect components separately: `mfrm_sim_1.item_effects` (a rater × item DataFrame) and `mfrm_sim_1.threshold_effects` (a rater × threshold DataFrame, zero-mean per threshold across raters -- i.e. each column averages to zero, matching what the bivector PAIR estimator can actually recover, rather than each row summing to zero). The reconstructed full rater × item × threshold matrix, in the same format as `MFRM_Sim_Matrix`'s, is also available as `mfrm_sim_1.facet_effects`, should you want it.

In [ ]:
mfrm_sim_1.item_effects.to_csv('mfrm_sim_1_item_effects.csv')
mfrm_sim_1.item_effects.head()

In [ ]:
mfrm_sim_1.threshold_effects.to_csv('mfrm_sim_1_threshold_effects.csv')
mfrm_sim_1.threshold_effects.head()

Save the generating person locations to file, and view the first 5 lines.

In [ ]:
mfrm_sim_1.persons.to_csv('mfrm_sim_1_persons.csv', header=None)
mfrm_sim_1.persons.head()

View `max_score`.

In [ ]:
mfrm_sim_1.max_score

Create an object `mfrm_1` of the class `MFRM` from the response dataframe for analysis. The new object `mfrm_1` automatically inherits all the parameters from `mfrm_sim_1`, storing them under a namespace `.generating`.

In [ ]:
mfrm_1 = rp.MFRM(mfrm_sim_1)

You may wish to create a simulation based on specified, known item locations and/or person locations. This may be done by passing lists to the `manual_items`, `manual_item_effects`, `manual_threshold_effects`, `manual_thresholds` and/or `manual_persons` arguments (in which case, there is no need to pass the relevant `item_range`, `item_facet_range`, `threshold_facet_range`, `category_base`, `max_disorder`, `person_sd` or `offset` arguments). You may also customise the names of the items and/or persons by passing lists of the correct length to the `manual_person_names` and/or `manual_item_names` arguments.

The `manual_items` and `manual_persons` arguments may also be used to generate random item locations and/or person locations according to distributions other than the default uniform (for items) and normal (for persons). This is what is done in the example `mfrm_sim_2` below: a set of specified, fixed item locations (4 items with locations between -1.5 and +1.5 logits and maximum score of 5) and a set of Rasch-Andrich thresholds (summing to zero) are passed together with 5 raters' item effects and threshold effects specified separately, and a random uniform distribution of person locations (between -2 and +2 logits). For this simulation, we also set a proportion of 10% missing data (missing completely at random) by passing the argument `missing=0.1`.

Rater_1 and Rater_4 have no threshold-shape effect at all (a flat `[0, 0, 0, 0, 0]`, i.e. matrix-like behaviour); Rater_2 and Rater_3 have opposite category-width profiles (each summing to zero, per the bivector model's identification constraint); Rater_5 has a milder version of Rater_3's profile. Independently, the item effects give Rater_3 a mild leniency, Rater_4 a stronger leniency, and Rater_5 a stronger severity, with Rater_1 and Rater_2 neutral.

In [ ]:
mfrm_sim_2 = rp.MFRM_Sim_Bivector(no_of_items=4,
                                   no_of_persons=500,
                                   no_of_facet_elements=5,
                                   max_score=5,
                                   missing=0.1,
                                   manual_persons=np.random.uniform(-2, 2, 500),
                                   manual_items=[-1.5, -0.5, 0.5, 1.5],
                                   manual_thresholds=[-2, -1, 0, 1, 2],
                                   manual_item_effects={'Rater_1': {'Item_1': 0, 'Item_2': 0, 'Item_3': 0, 'Item_4': 0},
                                                         'Rater_2': {'Item_1': 0, 'Item_2': 0, 'Item_3': 0, 'Item_4': 0},
                                                         'Rater_3': {'Item_1': -0.5, 'Item_2': -0.5, 'Item_3': -0.5, 'Item_4': -0.5},
                                                         'Rater_4': {'Item_1': -1, 'Item_2': -1, 'Item_3': -1, 'Item_4': -1},
                                                         'Rater_5': {'Item_1': 1, 'Item_2': 1, 'Item_3': 1, 'Item_4': 1}},
                                   manual_threshold_effects={'Rater_1': [0, 0, 0, 0, 0],
                                                              'Rater_2': [-1, -0.5, 0, 0.5, 1],
                                                              'Rater_3': [1, 0.5, 0, -0.5, -1],
                                                              'Rater_4': [0, 0, 0, 0, 0],
                                                              'Rater_5': [0.5, 0.25, 0, -0.25, -0.5]})

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_2.responses`, to file, and view the first 5 lines.

In [ ]:
mfrm_sim_2.responses.to_csv('mfrm_sim_2_responses.csv')
mfrm_sim_2.responses.head()

View the item locations and Rasch-Andrich thresholds (as specified above).

In [ ]:
mfrm_sim_2.items

In [ ]:
mfrm_sim_2.thresholds

View the item effects and threshold effects (as specified above).

In [ ]:
mfrm_sim_2.item_effects

In [ ]:
mfrm_sim_2.threshold_effects

View `max_score`.

In [ ]:
mfrm_sim_2.max_score

Create an object, `mfrm_2`, of the class `MFRM` from the response dataframe for analysis.

In [ ]:
mfrm_2 = rp.MFRM(mfrm_sim_2)

The two `MFRM` objects `mfrm_1` and `mfrm_2` are now available for analysis and, where appropriate, comparison of the recovered estmates with the generating estimates. See the example `Bivector MFRM` notebook for details on how to run an `MFRM` analysis.